<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
RAG: Grundlagen, Indexierung &amp; RAG-Chain
</b></font> </br></p>

---

Dieses Notebook bündelt die bisherigen Module M11 bis M13 zu einem durchgängigen Lernpfad:

1. RAG-Konzepte, Chunking und Embeddings
2. ChromaDB und Dokument-Indexierung
3. Retriever und vollständige RAG-Chain

Setup, Abhängigkeiten und Korpus-Download werden nur einmal ausgeführt. Die unterschiedlichen Chunking-Varianten und Collection-Namen aus M12 und M13 bleiben bewusst erhalten, damit ihre Auswirkungen vergleichbar bleiben.

Aufgaben und weiterführende Dokumente sind am Ende jeweils in einem gemeinsamen Abschnitt gebündelt.


**Beitrag zum Leitprojekt:** Ab hier entsteht das **Evidence Tool** des Meeting- & Briefing-Agent. RAG ist keine eigenständige Anwendung, sondern die kontrollierte Fähigkeit, Behauptungen aus Meeting-Protokollen, Entscheidungsvorlagen und Fachartikeln mit abrufbaren Belegen zu unterlegen — Chunking und Embeddings sind die technische Grundlage dafür, dass Mara Vogt Quellen später nachvollziehbar zitiert bekommt. Das ist der erste Baustein der **Handeln**-Fähigkeit im roten Faden des Kurses.

**Praxisanker:** Mara Vogt kann semantisch nach einer Entscheidung oder einem Risiko suchen, auch wenn ihre Frage nicht dieselben Wörter wie das Protokoll verwendet.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "m11-rag-konzepte"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "tags": ["m11", "rag"],
    "metadata": {"notebook": "m11", "version": "1.0"}
}


In [ ]:
#@title 📦 Installationen { display-mode: "form" }
install_packages([
                ('markitdown[all]', 'markitdown'),
                ('pypdf', 'pypdf'),
                'langchain_huggingface',
                ('unstructured[all-docs]>=0.11.2', 'unstructured'),
                ])

In [ ]:
#@title 📂 Meeting-Briefing-Korpus laden { display-mode: "form" }
import shutil
from pathlib import Path
from genai_lib.utilities import copy_from_github

KORPUS_QUELLE = "ralf-42/Agenten/02_daten/01_text/korpus_meeting_briefing"
KORPUS_MASKE = "*.pdf"
KORPUS_TARGET = "/content/files"

shutil.rmtree(KORPUS_TARGET, ignore_errors=True)
copy_from_github(
    source=KORPUS_QUELLE,
    target=KORPUS_TARGET,
    mask=KORPUS_MASKE,
)

pdfs = sorted(Path(KORPUS_TARGET).glob("*.pdf"))
print(f"Meeting-Briefing-Korpus geladen: {len(pdfs)} PDFs")
for pdf in pdfs[:5]:
    print("-", pdf.name)


# 1 | Überblick: RAG, Architektur und Nutzen
---

In [ ]:
from genai_lib.utilities import mermaid

# RAG-Architektur
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart TB
    DOC([Dokumente]) --> CHUNK[Chunking]
    CHUNK --> EMBED[Embedding-Modell]
    EMBED --> DB[(Vektordatenbank)]
    QUERY([Nutzeranfrage]) --> QEMBED[Embedding]
    QEMBED --> SEARCH[Similarity Search]
    DB --> SEARCH
    SEARCH --> KONTEXT[Kontext]
    QUERY --> PROMPT[Prompt]
    KONTEXT --> PROMPT
    PROMPT --> LLM[LLM]
    LLM --> ANT([Antwort])
"""
mermaid(diagram, width=700)

**RAG** (**R**etrieval-**A**ugmented **G**eneration) erweitert Sprachmodelle um **externes Wissen**.  
Für Mara bedeutet das: Der Assistant muss nicht aus dem Gedächtnis antworten, sondern kann relevante Stellen aus Meeting-Protokollen, Entscheidungsvorlagen und Fachartikeln abrufen und als Kontext verwenden.

In diesem Modul geht es um die Bausteine dafür:

1. Warum naive Suche nicht reicht
2. Wie Texte in Chunks zerlegt werden
3. Wie Embeddings semantische Nähe messbar machen
4. Wie daraus ein erster Mini-RAG-Loop entsteht



<p><font color='black' size="5">
Vergleich: LLM vs. LLM + RAG
</font></p>

| Eigenschaft | LLM allein | LLM + RAG |
|-------------|-----------|----------|
| Wissensbasis | Bis Trainings-Cutoff | Ständig aktualisierbar |
| Private Daten | ❌ Nicht vorhanden | ✅ Aus Vektordatenbank |
| Halluzinationen | Häufig bei unbekanntem Wissen | Reduziert durch Quellenbindung |
| Anpassbarkeit | Fine-Tuning notwendig | Neue Dokumente einfach hinzufügen |
| Transparenz | Schwer nachvollziehbar | Quellen sichtbar |



### Warum RAG?
---

<p><font color='black' size="5">
Das Problem: LLMs sind eingefroren
</font></p>


Mara sucht in Meeting-Protokollen und Projektunterlagen nicht nur nach exakten Stichwörtern. Typische Fragen lauten:

- "Welche Entscheidung zur Vektordatenbank gilt aktuell — D1 oder D2?"
- "Welchen Status hat Risiko R2 gerade?"
- "Welche offenen Punkte bestehen laut dem letzten Steuerkreis-Protokoll?"

Eine reine Volltextsuche findet dabei leicht zu viel, zu wenig oder die falschen Stellen. RAG löst dieses Problem, indem semantisch passende Textstellen gesucht und dem LLM als Kontext mitgegeben werden.



<p><font color='black' size="5">
Die RAG-Lösung
</font></p>


RAG trennt das **Reasoning** (im LLM) vom **Wissen** (im Korpus):

1. **Retrieval** – relevante Passagen aus dem Meeting-Briefing-Korpus abrufen
2. **Augmentation** – den Prompt mit diesen Passagen anreichern
3. **Generation** – eine Antwort auf Basis der Quellen erzeugen

**Wann RAG verwenden?**

| Szenario | RAG? | Begründung |
|---|---|---|
| Fachartikel durchsuchen | Ja | Wissen liegt im Korpus, nicht sicher im Modell |
| Allgemeine Smalltalk-Frage | Nein | Modellwissen reicht meist aus |
| Antwort mit Quellenpflicht | Ja | Quellen müssen abrufbar sein |
| Rechenaufgabe | Nein | Ein Tool ist besser geeignet |
| Out-of-Corpus-Frage | Nein | Assistant soll "Nicht im Korpus" sagen |


### RAG-Architektur
---


<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/rag_process.png" width="700" alt="Avatar">
<p><font color='black' size="2">
</font></p>

*RAG-Prozess: Von der Indexierung bis zur Antwort-Generierung*


<p><font color='black' size="5">
Einschränkungen von RAG
</font></p>

RAG ist kein Allheilmittel. Es hilft, wenn die Antwort im Korpus belegbar ist. Es hilft nicht, wenn:

- die passenden Dokumente fehlen,
- Retrieval irrelevante Chunks liefert,
- der Prompt Quellenpflicht nicht erzwingt,
- oder das System trotz fehlender Evidenz frei formuliert.

Für den Meeting- & Briefing-Agent ist deshalb wichtig: **Antworten brauchen Quellen oder den Hinweis "Nicht im Korpus".**


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [RAG-Pipeline](https://editor.p5js.org/ralf.bendig.rb/full/RrfB3nCwK)


# 2 | Kontext, Tokens und Chunking
---

Große Sprachmodelle verarbeiten Text als Tokens. Für RAG ist das wichtig, weil Frage, Kontext und Antwort gemeinsam in das Kontextfenster passen müssen.

Wenn zu viele oder zu lange Chunks in den Prompt gelangen, steigen Kosten und Latenz. Wenn zu wenige Chunks verwendet werden, fehlt eventuell die entscheidende Quelle.


**Tokenisierung: Die Grundlage der Textverarbeitung**

Tokenisierung ist der Prozess, bei dem Text in kleinere Einheiten, sogenannte Tokens, zerlegt wird. Diese Tokens können Wörter, Teilwörter oder sogar einzelne Zeichen sein.

**Warum ist Tokenisierung wichtig?**

1. **Einheitliche Verarbeitung:** LLMs haben eine begrenzte Eingabelänge. Die Tokenisierung stellt sicher, dass Texte in einem einheitlichen Format vorliegen, das vom Modell verarbeitet werden kann.

2. **Bedeutungserfassung:** Viele Tokens repräsentieren semantische Einheiten, was dem Modell hilft, die Bedeutung des Textes besser zu erfassen.

3. **Effiziente Verarbeitung:** Tokenisierte Texte können effizienter verarbeitet und in numerische Vektoren umgewandelt werden, was für die Eingabe in neuronale Netze notwendig ist.


**Hilfe:**

1 DIN A4 Seite hat ca. 300 Worte und ca. 450 Token

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [OpenAI Tokenizer](https://platform.openai.com/tokenizer)
- [GPT Tokenizer Playground](https://gpt-tokenizer.dev/)
- [TF Projector](https://projector.tensorflow.org/?hl=de)


### Chunking-Strategien
---

<p><font color='black' size="5">
Warum Chunking?
</font></p>

**Chunking: lange Texte in sinnvolle Abschnitte teilen**

Chunking zerlegt Protokolle und Fachartikel in kleinere, zusammenhängende Abschnitte. Für Mara ist das entscheidend: Der Assistant soll nicht ein ganzes Protokoll oder Paper in den Prompt legen, sondern nur die relevanten Passagen.

Gute Chunks sind:

1. klein genug für den Prompt,
2. groß genug für sinnvollen Kontext,
3. mit etwas Overlap gegen Informationsverlust an Grenzen geschützt.


**Strategien im Vergleich**

| Strategie | Vorteil | Nachteil | Einsatz im Kurs |
|---|---|---|---|
| Fixed-Size | Einfach, schnell | Zerreißt Sätze | Nur als Anti-Pattern |
| Satzbasiert | Lesbarer Kontext | Uneinheitliche Länge | Gut für kurze Texte |
| Recursive | Robust und praktikabel | Parameter nötig | Standard |
| Semantisch | Sehr passend | Teurer und langsamer | – |

**Empfehlung für den Start:** `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)`


[ChunkViz](https://chunkviz.up.railway.app/)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = (
    "Retrieval-Augmented Generation kombiniert Suche und Antwortgenerierung. "
    "Der Assistant ruft relevante Passagen aus einem Fachartikel-Korpus ab. "
    "Embeddings bilden Text als Vektoren ab. Dadurch können semantisch ähnliche Fragen "
    "und Dokumentstellen gefunden werden, auch bei unterschiedlichen Formulierungen. "
    "Chunking teilt lange Paper in kleinere Abschnitte. So bleibt der Kontext handhabbar "
    "und der Assistant kann Quellen präziser angeben."
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=40,
    length_function=len,
)

chunks = splitter.split_text(text)

print(f"Original: {len(text)} Zeichen")
print(f"Chunks:   {len(chunks)} Stück (chunk_size=220, overlap=40)")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} Zeichen) ---")
    print(chunk)
    print()


# 3 | Embeddings: Konzept und Demo
---


<p><font color='black' size="5">
Was sind Embeddings?
</font></p>


Ein **Embedding** ist eine numerische Darstellung von Text als Vektor in einem hochdimensionalen Raum.  
Semantisch ähnliche Texte liegen dabei **nah beieinander** im Vektorraum.

```
"Hund spielt"  →  [0.23, -0.11,  0.87, ...]   <- nah beieinander!
"Tier tobt"    →  [0.21, -0.09,  0.84, ...]
"Aktienkurs"   →  [-0.45, 0.72, -0.31, ...]   <- weit entfernt
```



<p><font color='black' size="5">
OpenAI Embedding-Modelle
</font></p>


| Modell | Dimensionen | Kosten | Empfehlung |
|--------|------------|--------|------------|
| `text-embedding-3-small` | 1.536 | Günstig | ✅ Standard |
| `text-embedding-3-large` | 3.072 | Höher | Anspruchsvolle Suche |
| `text-embedding-ada-002` | 1.536 | Mittel | ⚠️ Veraltet |





| Maß | Wertebereich | Empfehlung |
|-----|------------|------------|
| **Kosinus-Ähnlichkeit** | -1 bis 1 | ✅ OpenAI empfohlen |
| Euklidischer Abstand | 0 bis ∞ | Alternativ |

**Interpretation:** 1.0 = identisch  •  0.5 = verwandt  •  0.0 = unrelated  •  -1.0 = gegensätzlich

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [Embeddings 2D](https://editor.p5js.org/ralf.bendig.rb/full/LPjLkzWbE)
- [Embeddings 3D](https://editor.p5js.org/ralf.bendig.rb/full/gFBwB2E8n)
- [Kontextfenster](https://editor.p5js.org/ralf.bendig.rb/full/tLnUgyZRK)
- [TF Projector](https://projector.tensorflow.org/?hl=de)


### Embedding-Demo
---

Embeddings für drei Sätze werden erstellt und nach semantischer Ähnlichkeit verglichen:

- **Satz A & B** – Beide über Tiere → sollten **ähnlich** sein
- **Satz A & C** – Tier vs. Börse → sollten **unähnlich** sein

In [ ]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

embeddings_model = OpenAIEmbeddings(model=EMBEDDINGS)
print("✅ Embedding-Modell geladen: text-embedding-3-small (1536 Dimensionen)")

In [ ]:
# Drei Beispielsätze
satz_a = "Der Hund spielt im Park."
satz_b = "Das Tier tobt auf der Wiese."
satz_c = "Die Aktie fiel gestern um 3 Prozent."

vektoren = embeddings_model.embed_documents([satz_a, satz_b, satz_c])

def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

sim_ab = cosine_similarity(vektoren[0], vektoren[1])
sim_ac = cosine_similarity(vektoren[0], vektoren[2])

mprint(f"""
## Ergebnis: Semantische Ähnlichkeit

| Vergleich | Thema | Ähnlichkeit | Bewertung |
|-----------|-------|-------------|----------|
| A vs. B | Tier-Themen | {sim_ab:10.4f} | 🟢 Ähnlich |
| A vs. C | Tier vs. Börse | {sim_ac:10.4f} | 🔴 Unähnlich |

Vektordimensionen: {len(vektoren[0])}
""")

# 4 | Mini-RAG: Vom Chunk zur Antwort
---

Die vorherigen Abschnitte haben Embeddings und Chunking einzeln gezeigt.  
Jetzt kombinieren wir beides zu einem **Mini-RAG-Loop**: nicht mit allen PDFs, sondern mit kurzen Meeting-Briefing-Korpus-Auszügen.

**Ablauf:**

1. Kurze Auszüge aus Meeting-/Projekt-PDFs und Fachartikeln werden eingebettet
2. Der Retriever findet die ähnlichsten Auszüge
3. Der Prompt wird mit Kontext angereichert
4. Das LLM antwortet quellengebunden


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from genai_lib.model_config import WORKER
# Mini-Wissensdatenbank: kurze, kuratierte Auszüge zum Meeting-Briefing-Korpus.
mini_docs = [
    "Quelle: rag_survey_2312_10997.pdf | RAG kombiniert Retrieval, Augmentation und Generation, um externes Wissen in LLM-Antworten einzubinden.",
    "Quelle: rag_survey_2312_10997.pdf | Naive RAG ist die einfache Grundform; advanced und modular RAG ergänzen Schritte wie Query-Transformation, Re-Ranking und Post-Retrieval-Verarbeitung.",
    "Quelle: ragas_evaluation_2309_15217.pdf | RAG-Evaluation betrachtet unter anderem Antworttreue, Kontextrelevanz und Antwortrelevanz.",
    "Quelle: robust_rag_irrelevant_context_2310_01558.pdf | Irrelevanter Kontext kann RAG-Antworten verschlechtern und ist ein Robustheitsproblem.",
]

vectorstore = Chroma.from_texts(mini_docs, embedding=embeddings_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

llm = init_chat_model(WORKER)

rag_prompt = PromptTemplate.from_template("""Beantworte die Frage ausschließlich auf Basis des Kontexts.
Wenn der Kontext nicht reicht, sage: Nicht im Korpus.

Kontext:
{context}

Frage: {question}

Antwort mit Quellenhinweis (am Ende in eckigen Klammern, z. B. [dateiname.pdf]):""")

mini_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
print("Mini-RAG-Chain bereit (in-memory, Research-Snippets)")


In [ ]:
fragen = [
    "Was sind Bestandteile von RAG?",
    "Wie unterscheidet sich naive RAG von advanced RAG?",
    "Warum kann irrelevanter Kontext problematisch sein?",
]

for _i, (frage) in enumerate(fragen, 1):
    antwort = mini_rag_chain.invoke(frage, config={**run_cfg, "run_name": f"m11_kap8_mini_rag_chain_antwort_{_i}"})
    retrieved = retriever.invoke(frage, config={**run_cfg, "run_name": f"m11_kap8_retrieved_{_i}"})
    quellen = sorted({d.page_content.split(" | ")[0].replace("Quelle: ", "") for d in retrieved})
    abgerufen = " | ".join(d.page_content.split(" | ", 1)[1][:80] + "…" for d in retrieved)
    mprint(f"""
---
**Frage:** {frage}
**Quellen:** {", ".join(quellen)}
**Abgerufen:** {abgerufen}
**Antwort:** {antwort}
""")


In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(5)
show_trace("m11-rag-konzepte", limit=3, show_steps=True)

# 5 | ChromaDB: Setup, Indexierung und Betrieb

Im nächsten Abschnitt wird der zuvor erklärte RAG-Prozess als persistente ChromaDB-Wissensbasis umgesetzt.

ChromaDB läuft lokal, braucht keine externe Infrastruktur und lässt sich in wenigen Zeilen mit LangChain verbinden. Für den Meeting- & Briefing-Agent ist ChromaDB die erste Vektordatenbank: Mara stellt eine Frage, ChromaDB findet relevante Passagen aus Protokollen, Entscheidungsvorlagen und Fachartikeln.

<p><font color='black' size="5">
Warum ChromaDB?
</font></p>

| Feature | ChromaDB | Nutzen im Kurs |
|---|---|---|
| Lokaler Betrieb | Keine Cloud nötig | Colab- und Laptop-tauglich |
| Persistenz | Dateisystem/SQLite | Index kann wiederverwendet werden |
| Metadaten-Filter | Eingebaut | Suche auf bestimmte PDFs begrenzen |
| LangChain-Integration | Nahtlos | Direkt nutzbar |

<p><font color='black' size="5">
In diesem Modul
</font></p>

1. ChromaDB einrichten
2. PDF-Dokumente laden und chunken
3. Meeting-Briefing-Korpus indexieren
4. Ähnlichkeitssuche, Score und Metadaten-Filter testen


In [ ]:
#@markdown   <p><font size="4" color='green'>  ChromaDB Setup</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    subgraph Indexierung
        DOCS["📄 Dokumente"] --> LOADER["📥 Document Loader"]
        LOADER --> SPLITTER["✂️ Text Splitter"]
        SPLITTER --> EMBED["🔢 OpenAI Embeddings"]
        EMBED --> CHROMA[("🗄️ ChromaDB")]
    end
    subgraph Suche
        Q["❓ Query"] --> QE["🔢 Query Embed"]
        QE --> SIM["🔍 Similarity Search"]
        CHROMA --> SIM
        SIM --> RESULT["📋 Top-K Docs"]
    end
'''

mermaid(diagram, width=1100)

### ChromaDB Setup
---


ChromaDB kann auf zwei Arten betrieben werden: Im **In-Memory-Modus** existieren die Daten nur im RAM — geeignet für schnelle Demos und Tests, aber nach dem Prozessende verloren. Im **persistenten Modus** werden die Daten auf dem Dateisystem gespeichert: lokal direkt im Projektverzeichnis, in Google Colab über ein eingebundenes Google Drive, da das Colab-Dateisystem temporär ist und alle Daten beim Neustart verloren gehen.

**Wichtig für Google Colab:**     
Das Colab-Dateisystem ist temporär — alle Daten gehen beim Neustart verloren.  
Für echte Persistenz muss Google Drive eingebunden werden.


In [ ]:
import chromadb
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

CHROMA_DIR = "/content/chroma_m12"

# Embedding-Modell
embeddings = OpenAIEmbeddings(model=EMBEDDINGS)

# ChromaDB persistent initialisieren
vectorstore = Chroma(
    collection_name="meeting_briefing_korpus_m12",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

print(f"\nChromaDB initialisiert")
print(f"  Speicherort: {CHROMA_DIR}")
print(f"  Collection:  meeting_briefing_korpus_m12")
print(f"  Dokumente:   {vectorstore._collection.count()}")

### Dokumente indexieren
---

M12 legt die Baseline fest: `meeting_briefing_korpus_m12` mit `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)`.

📦 **Hinweis zum Modul `genai_lib.briefing_rag`:** Diese Collection bleibt bewusst lokal/ephemer und dient nur der Demonstration dieser Chunking-Variante. Die produktive Kette (Collection `meeting_briefing_korpus_m14`) wird auf Google Drive persistiert und steht als wiederverwendbares Modul `genai_lib.briefing_rag` zur Verfügung.



Das Indexieren umfasst drei Schritte:

1. **Laden** – Dokumente einlesen (Text, PDF, Markdown, ...)
2. **Chunken** – In kleinere Abschnitte aufteilen
3. **Einbetten** – Vektoren erstellen und in ChromaDB speichern


<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/rag_process_01.png" width="650" alt="Indexierungs-Pipeline: Dokumente → Text Splitter → Embeddings → ChromaDB">
<p><font color='black' size="2">

</font></p>

*Indexierungs-Pipeline: Dokumente → Text Splitter → Embeddings → ChromaDB*

<p><font color='black' size="5">
Dokument-Loader im Kurs
</font></p>

Für diesen Meeting-Briefing-Korpus verwenden wir bewusst nur PDFs und `pypdf.PdfReader`. Das ist robuster als generische Dateilader, weil deren Zusatzabhängigkeiten je nach Paketversion fehleranfällig sein können.

| Loader | Dateityp | Import | Verwendung |
|---|---|---|---|
| `pypdf.PdfReader` | `.pdf` | `pypdf` | Standard für `korpus_meeting_briefing/*.pdf` |

Andere Loader für Markdown, Word oder HTML sind nützlich, gehören hier aber nicht zum Kernpfad. Sie werden später bei Transferbeispielen oder eigenen Korpora relevant.


In [ ]:
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*triton.*")

from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_pdf_documents(directory_path: str):
    """Lädt ausschließlich PDFs aus dem Meeting-Briefing-Korpus.

    Bewusst keine generischen Dateilader: Deren Zusatzabhängigkeiten sind je nach
    Pydantic-Version fehleranfällig. pypdf.PdfReader reicht für den Kurskorpus.
    """
    documents = []
    for pdf_path in sorted(Path(directory_path).glob("*.pdf")):
        reader = PdfReader(str(pdf_path))
        for page_number, page in enumerate(reader.pages):
            documents.append(
                Document(
                    page_content=page.extract_text() or "",
                    metadata={
                        "source": str(pdf_path),
                        "page": page_number,
                    },
                )
            )
    return documents


# Dokumente laden
dokumente_raw = load_pdf_documents(KORPUS_TARGET)

# Text Splitter: grob genug für Einsteiger, klein genug für präzises Retrieval
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.split_documents(dokumente_raw)

print(f"Geladene PDF-Seiten: {len(dokumente_raw)}")
print(f"Chunks nach Splitting: {len(chunks)}")
for i, chunk in enumerate(chunks[:5]):
    quelle = chunk.metadata.get("source", "?").split("/")[-1]
    print(f"  Chunk {i+1}: {len(chunk.page_content)} Zeichen | Quelle: {quelle}")
if len(chunks) > 5:
    print(f"  ... und {len(chunks) - 5} weitere Chunks")


In [ ]:
# Chunks in ChromaDB indexieren — nur wenn Collection noch leer ist
BATCH_SIZE = 500  # ChromaDB-Maximum liegt je nach Version bei ca. 5.461; 500 ist robust.

count_vorher = vectorstore._collection.count()
if count_vorher == 0:
    for start in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[start:start + BATCH_SIZE]
        vectorstore.add_documents(batch)
        ende = min(start + BATCH_SIZE, len(chunks))
        print(f"  Batch {start // BATCH_SIZE + 1}: {ende}/{len(chunks)} Chunks indexiert")
    print(f"\nIndexiert: {vectorstore._collection.count()} Dokumente neu angelegt")
else:
    print(f"Collection hat bereits {count_vorher} Einträge — kein erneutes Indexieren")


### Collections verwalten
---



Eine **Collection** in ChromaDB ist eine benannte Gruppe von Dokumenten – vergleichbar mit einer Tabelle in SQL.



<p><font color='black' size="5">
Wichtige Operationen
</font></p>


| Operation | Methode | Beschreibung |
|-----------|---------|-------------|
| Dokument hinzufügen | `add_documents()` | Neue Chunks indexieren |
| Ähnlichkeitssuche | `similarity_search()` | Top-K ähnliche Dokumente |
| Mit Score | `similarity_search_with_score()` | Suche + Ähnlichkeitswert |
| Metadaten-Filter | `filter={...}` | Nur bestimmte Dokumente |
| Alle Dokumente | `get()` | Rohe Collection-Daten |

In [ ]:
from genai_lib.model_config import ROUTER
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langsmith import tracing_context

# Query-Rewrite und Retriever werden als zwei nachvollziehbare Schritte verbunden.
llm_rewrite = init_chat_model(ROUTER)
rewrite_prompt = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m12_query_rewrite_prompt.md",
    mode="T",
)
rewrite_chain = rewrite_prompt | llm_rewrite | StrOutputParser()
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

frage_raw = "Wie unterscheidet sich naive RAG von advanced RAG?"
with tracing_context(enabled=True):
    frage = rewrite_chain.invoke({"user_question": frage_raw}, config={**run_cfg, "run_name": "m12_kap4_frage"}).strip()
    ergebnisse = retriever.invoke(frage, config={**run_cfg, "run_name": "m12_kap4_ergebnisse"})

print(f"Originalfrage: {frage_raw}")
print(f"Retrieval-Query: {frage}")
print()
print("Top-2 Ergebnisse:")
for i, doc in enumerate(ergebnisse):
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    print(f"{i+1}. [{quelle}] {doc.page_content[:180]}...")


In [ ]:
# Suche mit Chroma-Distanzen (kleiner = ähnlicher)
ergebnisse_mit_distance = vectorstore.similarity_search_with_score(
    "Warum kann irrelevanter Kontext bei RAG problematisch sein?",
    k=3,
)

mprint("## Suchergebnis mit Distanzwerten\n")
rows = []
for doc, distance in ergebnisse_mit_distance:
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    preview = doc.page_content[:100].replace("\n", " ")
    rows.append(f"| {quelle} | {distance:.4f} | {preview}... |")

mprint(
    "| Quelle | Distanz (kleiner = ähnlicher) | Inhalt (Vorschau) |\n"
    "|---|---:|---|\n"
    + "\n".join(rows)
)


In [ ]:
# Metadaten-Filter: Nur Dokumente aus einer bestimmten PDF-Datei suchen
filter_quelle = "/content/files/protokoll_steuerkreis_2026-04-14.pdf"
gefiltert = vectorstore.similarity_search(
    "Welche Entscheidung wurde zur Vektordatenbank getroffen?",
    k=2,
    filter={"source": filter_quelle},
)

print("Gefilterte Suche:", filter_quelle.split("/")[-1])
for doc in gefiltert:
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    print(f"  - {quelle}: {doc.page_content[:140]}...")


# 7 | Retriever und Similarity Search

Die indexierte Wissensbasis wird nun über Retriever und eine vollständige LCEL-RAG-Chain nutzbar gemacht.



In *RAG-Konzepte & Embeddings* haben wir Embeddings verstanden, in *ChromaDB & Indexing* ChromaDB kennengelernt.
Jetzt **verbinden** wir alles: eine vollständige **RAG-Chain** mit LangChain.

**RAG-Chain vs. RAG-Agent**

| Eigenschaft | RAG-Chain | RAG-Agent |
|-------------|-----------|----------|
| Ablauf | Deterministisch (immer gleich) | Dynamisch (entscheidet selbst) |
| Tool-Nutzung | Nein | Ja, inklusive RAG als Tool |
| Kontrolle | Vollständig | Flexibel, aber weniger vorhersehbar |
| Einstieg | ✅ Einfacher | Komplexer |

**Was wir bauen**

Eine RAG-Chain, die:
1. Dokumente in ChromaDB indexiert (Phase 1)
2. Bei einer Anfrage relevante Chunks abruft (Retrieval)
3. Den LLM mit dem Kontext aufruft und antwortet (Generation)

In [ ]:
#@markdown   <p><font size="4" color='green'>  RAG-Flow</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    subgraph Setup["Setup (einmalig)"]
        DOCS["📄 Dokumente"] --> SPLIT["✂️ Splitter"]
        SPLIT --> EMBED["🔢 Embeddings"]
        EMBED --> DB[("🗄️ ChromaDB")]
    end

    subgraph Chain["RAG-Chain (LCEL)"]
        Q["❓ Frage"] --> RET["🔍 Retriever"]
        DB --> RET
        RET --> FMT["📝 format_docs()"]
        Q --> PROMPT["📋 RAG-Prompt"]
        FMT --> PROMPT
        PROMPT --> LLM["🤖 LLM"]
        LLM --> PARSE["✂️ StrOutputParser"]
        PARSE --> ANS["💬 Antwort"]
    end
'''

mermaid(diagram, width=1300)

### Retriever erstellen
---



Ein **Retriever** ist die Schnittstelle zwischen der Vektordatenbank und der Chain.  
Er empfängt eine Frage und gibt die relevantesten Dokumente zurück.

<p><font color='black' size="5">
Wissensbasis aufbauen
</font></p>

Der Meeting-Briefing-Korpus besteht aus PDFs. Für den stabilen Colab-Lauf wird deshalb direkt mit `pypdf.PdfReader` geladen und anschließend ohne zusätzliche Text-Splitter-Abhängigkeit in Chunks zerlegt.


M13 verwendet bewusst kleinere Character-Chunks (`300/30`), damit Retrieval-Treffer in der Chain-Demo kompakt und gut inspizierbar bleiben. Der Collection-Name trägt deshalb das Modul-Suffix `_m13`.

📦 **Hinweis zum Modul `genai_lib.briefing_rag`:** Diese Collection bleibt bewusst lokal/ephemer und dient nur der Demonstration dieser Chunking-Variante. Die produktive Kette (Collection `meeting_briefing_korpus_m14`) wird auf Google Drive persistiert und steht als wiederverwendbares Modul `genai_lib.briefing_rag` zur Verfügung.

In [ ]:
from pathlib import Path

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings

from genai_lib.model_config import WORKER
# LLM und Embeddings initialisieren
llm = init_chat_model(WORKER)
embeddings = OpenAIEmbeddings(model=EMBEDDINGS)


In [ ]:
def load_pdf_documents(directory_path: str):
    """Lädt alle PDF-Seiten aus einem Verzeichnis."""
    documents = []
    for pdf_path in sorted(Path(directory_path).glob("*.pdf")):
        try:
            reader = PdfReader(str(pdf_path))
            for page_number, page in enumerate(reader.pages):
                documents.append(
                    Document(
                        page_content=page.extract_text() or "",
                        metadata={
                            "source": str(pdf_path),
                            "page": page_number,
                        },
                    )
                )
        except Exception as exc:
            print(f"  ⚠ {pdf_path.name} übersprungen: {exc}")
    return documents


def split_documents_by_characters(documents, chunk_size=300, chunk_overlap=30):
    """Teilt Dokumentseiten in überlappende Zeichen-Chunks."""
    chunks = []
    step = max(1, chunk_size - chunk_overlap)
    for doc in documents:
        text = doc.page_content or ""
        for chunk_index, start in enumerate(range(0, len(text), step)):
            chunk_text = text[start:start + chunk_size].strip()
            if not chunk_text:
                continue
            metadata = dict(doc.metadata)
            metadata["chunk_index"] = chunk_index
            chunks.append(Document(page_content=chunk_text, metadata=metadata))
    return chunks


# Dokumente laden und splitten
dokumente_raw = load_pdf_documents(KORPUS_TARGET)
chunks = split_documents_by_characters(
    dokumente_raw,
    chunk_size=300,
    chunk_overlap=30,
)

# ChromaDB aufbauen (persistent) — mit Batching für große Korpora
BATCH_SIZE = 500
vectorstore = Chroma(
    collection_name="meeting_briefing_korpus_m13",
    embedding_function=embeddings,
    persist_directory="chroma_m13",
)

count_vorher = vectorstore._collection.count()
if count_vorher == 0:
    for start in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[start:start + BATCH_SIZE]
        vectorstore.add_documents(batch)
        ende = min(start + BATCH_SIZE, len(chunks))
        print(f"  Batch {start // BATCH_SIZE + 1}: {ende}/{len(chunks)} Chunks indexiert")
else:
    print(f"Collection hat bereits {count_vorher} Einträge — kein erneutes Indexieren")

print(f"Geladene PDF-Seiten: {len(dokumente_raw)}")
print(f"Chunks verfügbar:    {len(chunks)}")
print(f"Collection-Größe:    {vectorstore._collection.count()}")


### Similarity Search
---

Bevor wir die vollständige RAG-Chain bauen, testen wir den Retriever direkt. Hier geht es noch nicht um schöne Antworten, sondern um die Frage: **Findet der Retriever die passenden Passagen aus dem Meeting-Briefing-Korpus?**

| Parameter | Beschreibung | Startwert |
|---|---|---|
| `k` | Anzahl zurückgegebener Dokumente | 3 |
| `search_type` | Suchalgorithmus | `"similarity"` |
| Debugging | Treffer und Quellen ansehen | Pflicht vor RAG-Chain |


In [ ]:
# Retriever aus dem Vectorstore erstellen
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

# Testanfrage aus dem Eval-Set
test_frage = "Wie unterscheidet sich naive RAG von advanced RAG?"
gefundene_docs = retriever.invoke(test_frage, config={"run_name": "m13_kap3_gefundene_docs"})

print(f"Frage: {test_frage}\n")
print(f"Gefundene Dokumente: {len(gefundene_docs)}\n")
for i, doc in enumerate(gefundene_docs):
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    print(f"Dokument {i+1}:")
    print(f"  Quelle: {quelle}")
    print(f"  Inhalt: {doc.page_content[:180]}...")
    print()

In [ ]:
# Similarity Search mit Score (für Debugging)
ergebnisse = vectorstore.similarity_search_with_score(
    "Warum kann irrelevanter Kontext bei RAG problematisch sein?",
    k=3,
)

mprint("## Similarity Search mit Scores\n")
mprint("> **Score = Distanz:** Niedriger bedeutet ähnlicher. Chroma liefert hier keine Ähnlichkeit.")
rows = ["| Nr | Quelle | Score | Inhalt |", "|---:|---|---:|---|"]
for i, (doc, score) in enumerate(ergebnisse):
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    inhalt = doc.page_content[:90].replace("\n", " ")
    rows.append(f"| {i+1} | {quelle} | {score:.4f} | {inhalt}... |")
mprint("\n".join(rows))

# 8 | RAG-Chain mit LCEL
---



Aufbau einer vollständigen RAG-Chain mit **LCEL** (LangChain Expression Language).



<p><font color='black' size="5">
Chain-Architektur
</font></p>


```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
```

**Erklärung:**
1. `retriever | format_docs` – Holt Dokumente und formatiert sie als Text
2. `RunnablePassthrough()` – Leitet die Frage unverändert weiter
3. `rag_prompt` – Baut den Prompt mit Kontext und Frage
4. `llm` – Sendet an gpt-5.4-mini
5. `StrOutputParser()` – Extrahiert den Text aus der Antwort

In [ ]:
# RAG-Prompt aus Prompt-Template laden
rag_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m13_rag_prompt.md", mode="T")

# Hilfsfunktion: Dokumente mit Quellen formatieren
def format_docs(docs):
    teile = []
    for doc in docs:
        quelle = doc.metadata.get("source", "?").split("/")[-1]
        teile.append(f"Quelle: {quelle}\n{doc.page_content}")
    return "\n\n---\n\n".join(teile)

# RAG-Chain zusammenbauen (LCEL)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG-Chain bereit")

In [ ]:
# RAG-Chain testen
test_fragen = [
    "Welche drei Kernbestandteile eines RAG-Frameworks werden beschrieben?",
    "Wie unterscheidet sich naive RAG von advanced RAG?",
    "Warum kann irrelevanter Kontext bei RAG problematisch sein?",
]

antworten = []
for _i, (frage) in enumerate(test_fragen, 1):
    print(f"Frage: {frage}")
    antwort = rag_chain.invoke(frage, config={"run_name": f"m13_kap4_rag_chain_antwort_{_i}"})
    antworten.append(antwort)
    print(f"Antwort: {antwort}")
    print("-" * 60)
    print()


# 9 | LangSmith, Optimierung und Evaluation
---



LangSmith zeigt den vollständigen Trace der RAG-Chain – ideal zum Debuggen, wenn Antworten nicht passen.

**Was LangSmith anzeigt**

| Ebene | Inhalt | Nützlich für |
|-------|--------|-------------|
| **Chain-Ebene** | Input/Output der gesamten Chain | Gesamtbild |
| **Retriever-Ebene** | Welche Dokumente gefunden wurden | Retrieval-Qualität |
| **Prompt-Ebene** | Exakter Prompt (mit Kontext) | Prompt-Debugging |
| **LLM-Ebene** | Tokens, Kosten, Latenz | Performance |

**Trace mit run_name**

```python
antwort = rag_chain.invoke(frage, config={"run_name": "m13-kap5-rag-chain"})
```

In [ ]:
# RAG mit LangSmith-Tracing und run_name
run_cfg = {"run_name": "m13_kap5_rag_chain", "tags": ["rag", "m13", "meeting-briefing-korpus"]}

# Absichtlich ein schwieriger Fall: die Antwort steht im Korpus, das Retrieval findet sie aber nicht sicher (Erklärung unter der Zelle)
frage = "Welche Entscheidung (D1) traf der Steuerkreis am 17.03.2026 zur Vektordatenbank?"

from langsmith import tracing_context

with tracing_context(enabled=True):
    antwort = rag_chain.invoke(frage, config=run_cfg)

# Gefundene Dokumente separat anzeigen
kontext_docs = retriever.invoke(frage, config={"run_name": "m13_kap5_kontext_docs"})

mprint(f"""
## RAG-Ergebnis mit LangSmith-Trace

**Frage:** {frage}

**Verwendete Quellen ({len(kontext_docs)} Dokumente):**
""")

for i, doc in enumerate(kontext_docs):
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    print(f"  {i+1}. {quelle} – {doc.page_content[:100]}...")

mprint(f"""

**Antwort:**

{antwort}

> Trace verfügbar in LangSmith unter Projekt: m13-rag-chain
""")


**Was zeigt dieses Ergebnis?**

Die Frage ist im Korpus beantwortbar (`protokoll_steuerkreis_2026-03-17.pdf`), trotzdem lautet die Antwort „Nicht im Korpus.“ Der Grund liegt im **Retrieval**, nicht im Modell: Unter den drei abgerufenen Chunks ist in diesem Beispiel das Protokoll vom 17.03. nicht enthalten. Mögliche Ursachen:

- Die Chunks sind klein (`300/30`), und ein Protokoll wird in viele ähnliche Abschnitte zerlegt.
- Mehrere Steuerkreis-Protokolle behandeln ähnliche Themen und konkurrieren um die ersten Plätze.
- Datum und Kürzel (`D1`) bildet ein Embedding nur schwach ab.

Das Modell verhält sich dabei richtig: Es erfindet nichts, sondern meldet „Nicht im Korpus“, weil der Kontext nicht ausreicht. Im **LangSmith-Trace** (Retriever-Ebene) lässt sich das nachvollziehen. Kapitel 6 zeigt Gegenmaßnahmen (`k`, Chunk-Größe, Score-Threshold).


### RAG-Optimierung
---



*Häufige Probleme und Lösungen*

| Problem | Symptom | Lösung |
|---------|---------|--------|
| Falsche Dokumente | Irrelevante Antworten | Chunk-Größe reduzieren, k erhöhen |
| Halluzinationen | Fakten nicht in Quellen | Temperatur auf 0.0, strengere Prompts |
| Langsame Antworten | Hohe Latenz | Weniger Chunks (k), kleinere Modelle |
| Zu viel Kontext | Token-Limit | Komprimierung, k reduzieren |
| Inkonsistente Antworten | Unterschiedliche Resultate | temperature=0.0 setzen |

**Optimierungsstrategie**

```
1. Mit kleinem k starten (k=3)
2. LangSmith-Traces analysieren
3. Retrieval-Qualität prüfen (werden relevante Docs gefunden?)
4. Prompt anpassen (präziser, Quellenbindung erzwingen)
5. chunk_size experimentieren (200–500)
```

In [ ]:
# Optimierter Retriever mit Score-Threshold
# Hinweis: Der Schwellwert bezieht sich auf einen Relevanz-Score (höher = besser), nicht auf die Distanz aus Kapitel 3
retriever_opt = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 4},
)

# Optimierte RAG-Chain
rag_chain_opt = (
    {"context": retriever_opt | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Vergleich: Standard vs. optimiert
frage_test = "Welche Umsatzzahlen erzielte die Nordlicht Software GmbH im letzten Geschäftsjahr?"

docs_std = retriever.invoke(frage_test, config={"run_name": "m13_kap6_docs_std"})
docs_opt = retriever_opt.invoke(frage_test, config={"run_name": "m13_kap6_docs_opt"})

print(f"Standard Retriever (k=3): {len(docs_std)} Dokumente")
print(f"Optimierter Retriever (k=4, threshold=0.3): {len(docs_opt)} Dokumente")
print("Erwartung bei Out-of-Corpus-Fragen: lieber wenige/keine Treffer und klare Unsicherheit.")


In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("m13-rag-chain", limit=3, show_steps=True)

# A | Aufgaben
---

Die folgenden Aufgaben führen vom Verständnis semantischer Suche über die Indexierung bis zur vollständigen RAG-Chain. Die Aufgabenstellungen bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:** Generative KI kann als Unterstützung beim Verstehen und Debuggen eingesetzt werden. Die Lösung sollte anschließend selbst erklärt und geprüft werden.

**Grundlagen**
Erstelle Embeddings für einen kleinen Korpus mit **6-9 Sätzen aus 3 Themen**: RAG-Grundlagen, RAG-Evaluation und Robustheit (wie in den Mini-RAG-Auszügen aus Kapitel 8).

- `embeddings_model.embed_documents([...])` auf alle Sätze anwenden.
- Die Funktion `cosine_similarity(v1, v2)` aus Kapitel 7 verwenden.
- Kosinus-Ähnlichkeiten für mindestens 3 Satzpaare berechnen.
- Vektoren in `meine_vektoren` und Werte in `sim_ergebnisse` speichern.


In [ ]:
# Grundlagen: Embeddings + Kosinus-Ähnlichkeit
# Startpunkt: embeddings_model und cosine_similarity(v1, v2) aus Kapitel 7

# 1. 6-9 Sätze aus 3 Themen definieren (RAG-Grundlagen, RAG-Evaluation, Robustheit)
# 2. embeddings_model.embed_documents() auf alle Sätze anwenden, Vektoren in meine_vektoren speichern
# 3. cosine_similarity() für mindestens 3 Satzpaare berechnen, Werte in sim_ergebnisse sammeln


**Aufbau**
Untersuche, ob Embeddings Sätze desselben Themas näher zusammenlegen als Sätze verschiedener Themen.

- Die Sätze aus den Grundlagen nach Themen gruppieren (z. B. `rag_grundlagen`, `evaluation`, `robustheit`).
- Ähnlichkeiten innerhalb eines Themas (intra) und zwischen Themen (inter) vergleichen.
- Die Ergebnisse als Tabelle ausgeben: Paar, Typ, Ähnlichkeit.
- Prüfen: Sind Sätze zu RAG-Grundlagen untereinander ähnlicher als zu Robustheit oder Evaluation?


In [ ]:
# Aufbau: Intra- vs. Inter-Thema-Vergleich
# Startpunkt: embeddings_model, cosine_similarity() und die Sätze aus den Grundlagen

# 1. Sätze nach Themen gruppieren (saetze_nach_thema, z. B. rag_grundlagen, evaluation, robustheit)
# 2. Je Thema Embeddings berechnen (thema_vektoren)
# 3. Intra-Thema- und Inter-Thema-Kosinus-Ähnlichkeiten berechnen
# 4. Ergebnisse als Liste von Dicts (paar, typ, aehnlichkeit) in aufbau_ergebnisse sammeln und als Tabelle ausgeben


**Vertiefung**
Vergleiche `text-embedding-3-small` mit `text-embedding-3-large` anhand eines Satzpaars zu RAG-Evaluation.

- Ein zweites Modell `embed_large` erstellen und dieselbe Ähnlichkeit wie mit `embeddings_model` (small) berechnen.
- Beurteilen, ob sich die semantische Nähe sichtbar verändert.
- In 2-3 Sätzen erklären, wann sich der höhere Aufwand von `-large` für den Meeting- & Briefing-Agent lohnt (Kapitel 6).


In [ ]:
# Vertiefung: small vs. large Embedding-Modelle
# Startpunkt: embeddings_model (small) und cosine_similarity() aus Kapitel 7

# 1. embed_large = OpenAIEmbeddings(model="text-embedding-3-large")
# 2. Ein Satzpaar zu RAG-Evaluation mit beiden Modellen einbetten, Ähnlichkeit berechnen (sim_small, sim_large)
# 3. In embedding_diskussion (>= 2 Sätze) einordnen, wann sich -large für den Meeting- & Briefing-Agent lohnt


### Aufgabenblock: ChromaDB und Indexierung

<p><font color='black' size="5">
Meeting-Briefing-Korpus indexieren
</font></p>

Baue eine ChromaDB-Wissensdatenbank für die Meeting-/Projekt-PDFs und Fachartikel auf. Der Fokus liegt nicht auf vielen Features, sondern auf einem nachvollziehbaren Index: laden, chunken, suchen, filtern.


**Grundlagen**

1. Erstelle eine ChromaDB-Collection `briefing_demo`.
2. Indexiere mindestens 2 kurze Research-Snippets als `Document`-Objekte.
3. Teste eine Suchanfrage zu RAG-Grundlagen.


In [ ]:
# Grundlagen: ChromaDB-Collection + 2 Research-Snippets
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
collection = Chroma(collection_name="briefing_demo", embedding_function=embeddings)

# 1. Mindestens 2 Document-Objekte mit Research-Snippets erstellen
# 2. collection.add_documents(...) aufrufen
# 3. Suchanfrage zu RAG-Grundlagen mit collection.similarity_search() testen

**Aufbau**

1. Nutze den geladenen Meeting-Briefing-Korpus aus `/content/files/*.pdf`.
2. Konfiguriere den `RecursiveCharacterTextSplitter` explizit (`chunk_size=500`, `chunk_overlap=50`).
3. Indexiere die Chunks in eine persistente ChromaDB-Collection (`vectorstore`).
4. Führe 3 Suchanfragen mit unterschiedlichen `k`-Werten durch.
5. Zeige mindestens eine Suche mit Metadaten-Filter auf eine konkrete PDF-Datei.
6. Stelle die Ergebnisse als Markdown-Tabelle dar.


In [ ]:
# Aufbau: Meeting-Briefing-Korpus + Chunking + k-Vergleich
from pathlib import Path
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. PDFs aus KORPUS_TARGET laden (pypdf.PdfReader)
# 2. RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50) anwenden
# 3. Chunks in eine persistente ChromaDB-Collection (vectorstore) indexieren
# 4. Drei Suchanfragen mit unterschiedlichen k-Werten durchfuehren (results)
# 5. Ergebnisse als Markdown-Tabelle darstellen

**Vertiefung**

- Nutze Metadaten-Filter, um die Suche auf eine bestimmte PDF-Quelle einzuschränken.
- Vergleiche Treffer mit und ohne Filter.
- Verwende den Retriever aus dem Beispiel und aktiviere für diesen Referenzlauf
  `tracing_context`; analysiere anschließend den LangSmith-Trace.


In [ ]:
# Vertiefung: Metadaten-Filter auf PDF-Quelle
# Startpunkt: vectorstore aus dem Aufbau-Teil

# 1. Eine konkrete PDF-Quelle wählen:
#    quelle = "/content/files/<datei>.pdf"
# 2. Ungefilterte Suche durchführen und Treffer vergleichen.
# 3. Einen Retriever mit search_kwargs={"filter": {"source": quelle}} anlegen.
# 4. Den Retriever mit config=run_cfg in einem tracing_context-Kontext aufrufen.
# 5. Treffer aus beiden Suchen und den LangSmith-Trace vergleichen.


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [RAG-Pipeline](https://editor.p5js.org/ralf.bendig.rb/full/RrfB3nCwK)
- [Embeddings 2D](https://editor.p5js.org/ralf.bendig.rb/full/LPjLkzWbE)
- [Embeddings 3D](https://editor.p5js.org/ralf.bendig.rb/full/gFBwB2E8n)


### Aufgabenblock: Retriever und RAG-Chain

**Grundlagen**
- Bereits geladener Meeting-Briefing-Korpus aus `korpus_meeting_briefing/*.pdf` als Wissensbasis.
- Die persistente ChromaDB-Collection `meeting_briefing_korpus_m13` aus Kapitel 2 verwenden (`vectorstore`).
- Retriever mit einer einfachen Frage wie `"Wer ist Projektleiterin von Projekt Kompass?"` testen.
- Pro Treffer Dateiname und kurzer Ausschnitt anzeigen.


In [ ]:
# Grundlagen: Retriever aus ChromaDB aufbauen
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 1. retriever.invoke() mit einer einfachen Frage testen (docs)
# 2. Pro Treffer Dateiname und kurzen Ausschnitt ausgeben

**Aufbau**
- Mit einem eigenen Retriever (anderes `k`, z. B. 5) und dem Standard-RAG-Prompt aus `05_prompt/m13_rag_prompt.md` eine neue RAG-Chain bauen (`format_docs` aus Kapitel 4 verwenden).
- Diese drei Fragen beantworten: „Wer ist Projektleiterin von Projekt Kompass?“, „Wie unterscheidet sich naive RAG von advanced RAG?“ und „Welche Entscheidung (D1) traf der Steuerkreis am 17.03.2026 zur Vektordatenbank?“ (oder eigene Fragen zum Meeting-Briefing-Korpus).
- Pro Frage die verwendeten Quellen anzeigen.
- Beobachten, ob das größere `k` die dritte Frage (Fehlschlag aus Kapitel 5) beantwortbar macht.
- Retriever in `retriever` und Chain in `rag_chain` speichern.


In [ ]:
# Aufbau: RAG-Chain mit eigenem k und drei Briefing-Fragen
# Startpunkt: vectorstore, rag_prompt und format_docs aus den Kapiteln 2 und 4

# 1. retriever mit anderem k aus dem vectorstore bauen (z. B. k=5)
# 2. rag_chain per LCEL bauen: {"context": retriever | format_docs, "question": RunnablePassthrough()} | rag_prompt | llm | StrOutputParser()
# 3. Drei Fragen mit rag_chain.invoke() beantworten, Quellen pro Frage anzeigen
# 4. Kurz festhalten, ob die Entscheidungsfrage (17.03.2026) mit größerem k beantwortet wird


**Vertiefung**
- LangSmith-Tracing mit `run_name` und `tags` ergänzen.
- In LangSmith analysieren, welche Dokumente bei welcher Frage gefunden wurden.
- In 2 Sätzen begründen, welche Dokument-Chunks am häufigsten abgerufen werden und warum.
- Mindestens eine Out-of-Corpus-Frage testen, bei der die Antwort "Nicht im Korpus" sein sollte.


In [ ]:
# Vertiefung: LangSmith-Tracing + Retrieval-Analyse
# Startpunkt: retriever, rag_chain aus Aufbau

# 1. Mehrere Fragen (inkl. einer Out-of-Corpus-Frage) mit rag_chain.invoke(..., config=...) beantworten
# 2. Abgerufene Dokumente pro Frage in LangSmith prüfen
# 3. Beobachtung zu häufig abgerufenen Chunks in retrieval_beobachtung (str) festhalten

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [RAG-Pipeline](https://editor.p5js.org/ralf.bendig.rb/full/RrfB3nCwK)
- [OpenAI Tokenizer](https://platform.openai.com/tokenizer)


# B | Dokumente zum Weiterlesen
---

### Grundlagen, RAG und Embeddings

Ergänzende Artikel aus der Kurs-Dokumentation:

- [RAG-Konzepte](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/rag-konzepte.html)
- [Embeddings](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/embeddings.html)
- [Tokenizing & Chunking](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/tokenizing-chunking.html)
- [Context Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/context-engineering.html)
- [Einsteiger ChromaDB](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-chromadb.html)

### ChromaDB und Indexierung

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Einsteiger ChromaDB](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-chromadb.html)
- [Embeddings](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/embeddings.html)
- [Tokenizing & Chunking](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/tokenizing-chunking.html)
- [RAG-Konzepte](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/rag-konzepte.html)
- [Context Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/context-engineering.html)

### Retriever und RAG-Chain

Ergänzende Artikel aus der Kurs-Dokumentation:

- [RAG-Konzepte](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/rag-konzepte.html)
- [Context Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/context-engineering.html)
- [Einsteiger ChromaDB](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-chromadb.html)
- [LangChain Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langchain-best-practices.html)
- [Embeddings](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/embeddings.html)